**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 온톨로지 RAG: Apache AGE 기반 지식 그래프 추론 시스템

## 학습 목표

1️⃣ 온톨로지의 개념과 구성 요소를 이해한다  
2️⃣ 벡터 RAG, 그래프 RAG, 온톨로지 RAG의 차이를 비교한다  
3️⃣ Apache AGE의 특징과 기본 사용법을 학습한다  
4️⃣ 도메인 온톨로지를 설계하고 AGE 그래프로 구축한다  
5️⃣ 온톨로지 기반 추론과 LLM 연동을 실습한다  
6️⃣ 의료/법률/금융 도메인 활용 사례를 살펴본다  

---

### 실습 환경
- **GPU**: 선택사항
- **필수 패키지**: psycopg2, networkx, openai, langchain
- **외부 서비스**: PostgreSQL + Apache AGE 확장

In [ ]:
# 패키지 설치
!pip install -q psycopg2-binary networkx matplotlib openai langchain langchain-openai rdflib

In [ ]:
# 패키지 임포트 및 환경 설정
import os
import json
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional

# os.environ["OPENAI_API_KEY"] = "your-api-key-here"
# PostgreSQL + AGE 연결 정보
# DB_CONFIG = {
#     "host": "localhost",
#     "port": 5432,
#     "dbname": "agedb",
#     "user": "postgres",
#     "password": "password"
# }

print("패키지 로드 완료")

---

## 1️⃣ 온톨로지란?

### 개념

온톨로지(Ontology)는 특정 도메인의 **개념, 속성, 관계를 형식적으로 정의**한 지식 표현 체계입니다.  
단순한 지식 그래프와 달리, 온톨로지는 **스키마 레벨의 의미론적 구조**를 제공합니다.

```
지식 그래프:  (이재용) --[CEO]--> (삼성전자)           ← 인스턴스 레벨
온톨로지:     (Person) --[worksAt]--> (Organization)   ← 스키마 레벨
              + 제약조건: Person은 최대 1개의 Organization에 worksAt 가능
              + 추론규칙: worksAt의 역관계는 hasEmployee
```

### 핵심 구성 요소

| 구성 요소 | 설명 | 예시 |
|-----------|------|------|
| **클래스 (Class)** | 개념의 범주 | Person, Organization, Product |
| **프로퍼티 (Property)** | 클래스의 속성 | name, age, founded_year |
| **관계 (Relation)** | 클래스 간 연결 | worksAt, produces, locatedIn |
| **인스턴스 (Instance)** | 클래스의 구체적 개체 | 이재용 (Person), 삼성전자 (Organization) |
| **제약조건 (Constraint)** | 관계의 규칙 | 카디널리티, 도메인/레인지 |

### OWL / RDF 소개

- **RDF (Resource Description Framework)**: 트리플 (주어-술어-목적어) 기반 데이터 모델
- **RDFS**: RDF 스키마, 클래스와 프로퍼티의 계층 구조 정의
- **OWL (Web Ontology Language)**: 더 풍부한 표현력 (동치, 제약, 추론 규칙 등)

```turtle
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:Samsung rdf:type ex:Organization .
ex:Samsung ex:produces ex:Galaxy .
ex:JYLee rdf:type ex:Person .
ex:JYLee ex:worksAt ex:Samsung .
```

In [ ]:
# Python에서 간단한 온톨로지 정의

class OntologySchema:
    """도메인 온톨로지 스키마를 정의하는 클래스"""

    def __init__(self, name: str):
        self.name = name
        self.classes = {}       # 클래스 정의
        self.relations = {}     # 관계 정의
        self.constraints = []   # 제약조건

    def add_class(self, class_name: str, parent: str = None, properties: Dict = None):
        """클래스 정의 추가"""
        self.classes[class_name] = {
            "parent": parent,
            "properties": properties or {}
        }

    def add_relation(self, name: str, domain: str, range_cls: str,
                     cardinality: str = "many-to-many", inverse: str = None):
        """관계 정의 추가"""
        self.relations[name] = {
            "domain": domain,
            "range": range_cls,
            "cardinality": cardinality,
            "inverse": inverse
        }

    def add_constraint(self, description: str):
        """제약조건 추가"""
        self.constraints.append(description)

    def display(self):
        """온톨로지 스키마 출력"""
        print(f"=== 온톨로지: {self.name} ===")
        print(f"\n[클래스] ({len(self.classes)}개)")
        for cls_name, cls_info in self.classes.items():
            parent = f" (상위: {cls_info['parent']})" if cls_info['parent'] else ""
            print(f"  - {cls_name}{parent}")
            for prop, ptype in cls_info['properties'].items():
                print(f"      .{prop}: {ptype}")

        print(f"\n[관계] ({len(self.relations)}개)")
        for rel_name, rel_info in self.relations.items():
            inv = f" (역: {rel_info['inverse']})" if rel_info['inverse'] else ""
            print(f"  - {rel_info['domain']} --[{rel_name}]--> {rel_info['range']}  [{rel_info['cardinality']}]{inv}")

        if self.constraints:
            print(f"\n[제약조건] ({len(self.constraints)}개)")
            for c in self.constraints:
                print(f"  - {c}")


# IT 기업 도메인 온톨로지 정의
ontology = OntologySchema("한국 IT 기업 온톨로지")

# 클래스 정의
ontology.add_class("Entity", properties={"name": "string", "description": "string"})
ontology.add_class("Organization", parent="Entity", properties={"founded_year": "int", "industry": "string"})
ontology.add_class("Company", parent="Organization", properties={"revenue": "float", "employees": "int"})
ontology.add_class("Person", parent="Entity", properties={"birth_year": "int", "role": "string"})
ontology.add_class("Product", parent="Entity", properties={"category": "string", "release_year": "int"})
ontology.add_class("Technology", parent="Entity", properties={"domain": "string"})
ontology.add_class("Location", parent="Entity", properties={"country": "string", "city": "string"})

# 관계 정의
ontology.add_relation("worksAt", "Person", "Organization", "many-to-one", inverse="hasEmployee")
ontology.add_relation("produces", "Company", "Product", "one-to-many", inverse="producedBy")
ontology.add_relation("uses", "Product", "Technology", "many-to-many")
ontology.add_relation("locatedIn", "Organization", "Location", "many-to-one")
ontology.add_relation("competesWith", "Company", "Company", "many-to-many")
ontology.add_relation("subsidiaryOf", "Company", "Company", "many-to-one", inverse="hasSubsidiary")

# 제약조건
ontology.add_constraint("Company는 반드시 하나의 Location에 locatedIn 관계를 가져야 한다")
ontology.add_constraint("Person의 worksAt 관계는 최대 1개의 Organization으로 제한된다")
ontology.add_constraint("competesWith 관계는 대칭적이다 (A competesWith B이면 B competesWith A)")

ontology.display()

---

## 2️⃣ 벡터 RAG vs 그래프 RAG vs 온톨로지 RAG

### 구조적 차이 비교

| 특성 | 벡터 RAG | 그래프 RAG | 온톨로지 RAG |
|------|----------|------------|-------------|
| **데이터 구조** | 벡터 임베딩 | 지식 그래프 (트리플) | 온톨로지 + 지식 그래프 |
| **스키마** | 없음 | 암묵적 | 명시적 (OWL/RDF) |
| **검색 방식** | 코사인 유사도 | 그래프 탐색 | 온톨로지 추론 + 그래프 탐색 |
| **추론 능력** | 제한적 | 경로 기반 추론 | 논리적 추론 (상속, 전이, 대칭) |
| **관계 표현** | 암묵적 | 명시적 (레이블) | 형식적 (도메인/레인지/제약) |
| **장점** | 구축 쉬움, 범용적 | 관계 추론 | 도메인 전문 추론, 일관성 보장 |
| **단점** | 관계 손실 | 스키마 부재 | 구축 비용 높음 |
| **적합 도메인** | 일반 QA | 관계 질문 | 의료, 법률, 금융 등 전문 도메인 |

### 추론 능력 비교 예시

```
질문: "반도체를 생산하는 회사의 CEO가 졸업한 대학은?"

벡터 RAG:
  → 유사한 문서 검색 → 단일 홉 답변만 가능 → 실패 가능성 높음

그래프 RAG:
  → (반도체) ←[생산]← (삼성전자) ←[CEO]← (이재용) →[졸업]→ (서울대)
  → 경로 추적으로 답변 가능

온톨로지 RAG:
  → 온톨로지: produces의 domain은 Company, CEO의 range는 Person
  → 추론: Company.produces(Product) ∧ Person.ceoOf(Company) ∧ Person.graduatedFrom(University)
  → 스키마 기반 정확한 쿼리 생성 + 결과 검증
```

In [ ]:
# 세 가지 RAG 접근법 비교 시각화

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. 벡터 RAG
ax1 = axes[0]
ax1.set_title("Vector RAG", fontsize=14, fontweight="bold")
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
import numpy as np
np.random.seed(42)
x = np.random.uniform(2, 8, 15)
y = np.random.uniform(2, 8, 15)
ax1.scatter(x, y, s=100, c="#4ECDC4", alpha=0.7)
ax1.scatter([5], [5], s=200, c="#FF6B6B", marker="*", zorder=5, label="Query")
for i in range(3):
    ax1.plot([5, x[i]], [5, y[i]], "--", color="gray", alpha=0.5)
ax1.set_xlabel("Embedding Dim 1")
ax1.set_ylabel("Embedding Dim 2")
ax1.legend()

# 2. 그래프 RAG
ax2 = axes[1]
ax2.set_title("Graph RAG", fontsize=14, fontweight="bold")
G2 = nx.DiGraph()
G2.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("A", "E"), ("E", "F"), ("B", "F")])
pos2 = nx.spring_layout(G2, seed=42)
nx.draw(G2, pos2, ax=ax2, with_labels=True, node_color="#4ECDC4",
        node_size=800, font_size=12, arrows=True, arrowsize=15, edge_color="gray")

# 3. 온톨로지 RAG
ax3 = axes[2]
ax3.set_title("Ontology RAG", fontsize=14, fontweight="bold")
G3 = nx.DiGraph()
# 스키마 레벨 (상단)
G3.add_edges_from([("Person", "Org"), ("Org", "Product"), ("Org", "Location")])
# 인스턴스 레벨 (하단)
G3.add_edges_from([("p1", "o1"), ("o1", "pr1"), ("o1", "loc1")])
# 스키마-인스턴스 연결
G3.add_edges_from([("p1", "Person"), ("o1", "Org"), ("pr1", "Product"), ("loc1", "Location")])
pos3 = {
    "Person": (1, 4), "Org": (3, 4), "Product": (5, 4), "Location": (3, 5.5),
    "p1": (1, 1.5), "o1": (3, 1.5), "pr1": (5, 1.5), "loc1": (3, 0)
}
schema_nodes = ["Person", "Org", "Product", "Location"]
instance_nodes = ["p1", "o1", "pr1", "loc1"]
nx.draw_networkx_nodes(G3, pos3, nodelist=schema_nodes, node_color="#FF6B6B", node_size=800, ax=ax3)
nx.draw_networkx_nodes(G3, pos3, nodelist=instance_nodes, node_color="#4ECDC4", node_size=600, ax=ax3)
nx.draw_networkx_labels(G3, pos3, font_size=9, ax=ax3)
nx.draw_networkx_edges(G3, pos3, ax=ax3, arrows=True, arrowsize=12, edge_color="gray")
ax3.text(0.5, 6.2, "Schema Level", fontsize=10, color="#FF6B6B", fontweight="bold")
ax3.text(0.5, 2.5, "Instance Level", fontsize=10, color="#4ECDC4", fontweight="bold")

plt.tight_layout()
plt.savefig("rag_comparison.png", dpi=100, bbox_inches="tight")
plt.show()
print("세 가지 RAG 방식의 구조적 차이를 시각화했습니다.")

---

## 3️⃣ Apache AGE 소개

### Apache AGE란?

**Apache AGE (A Graph Extension)**는 PostgreSQL의 그래프 데이터베이스 확장입니다.

| 특성 | 설명 |
|------|------|
| **기반** | PostgreSQL 확장 (Extension) |
| **쿼리 언어** | openCypher (Neo4j 호환) |
| **장점** | SQL + Cypher 동시 사용, ACID 트랜잭션 |
| **비용** | 오픈소스 (Apache License 2.0) |
| **성능** | PostgreSQL 최적화 엔진 활용 |

### 왜 Apache AGE인가?

```
Neo4j 대비 Apache AGE의 장점:
  1. PostgreSQL 생태계 활용 (기존 인프라 재사용)
  2. SQL과 Cypher를 함께 사용 가능
  3. 완전한 오픈소스 (Neo4j Enterprise는 유료)
  4. 관계형 데이터 + 그래프 데이터 통합 관리
  5. ACID 트랜잭션 보장
```

### 설치 방법

```bash
# Docker로 Apache AGE 설치 (가장 간단)
docker pull apache/age
docker run -p 5432:5432 -e POSTGRES_PASSWORD=password apache/age

# 또는 PostgreSQL에 직접 설치
# sudo apt-get install postgresql-16-age
```

---

## 4️⃣ Apache AGE 설치 및 기본 사용법

### AGE 초기화 및 그래프 생성

In [ ]:
# Apache AGE 연결 및 관리 클래스

import psycopg2


class ApacheAGE:
    """Apache AGE 그래프 데이터베이스 관리 클래스"""

    def __init__(self, host="localhost", port=5432, dbname="agedb",
                 user="postgres", password="password"):
        self.conn = psycopg2.connect(
            host=host, port=port, dbname=dbname,
            user=user, password=password
        )
        self.conn.autocommit = True
        self.cursor = self.conn.cursor()
        self._init_age()
        print(f"Apache AGE 연결 성공: {host}:{port}/{dbname}")

    def _init_age(self):
        """AGE 확장 로드"""
        self.cursor.execute("CREATE EXTENSION IF NOT EXISTS age;")
        self.cursor.execute("LOAD 'age';")
        self.cursor.execute("SET search_path = ag_catalog, '$user', public;")

    def create_graph(self, graph_name: str):
        """그래프 생성"""
        try:
            self.cursor.execute(f"SELECT create_graph('{graph_name}');")
            print(f"그래프 '{graph_name}' 생성 완료")
        except psycopg2.errors.InvalidParameterValue:
            print(f"그래프 '{graph_name}'이 이미 존재합니다")

    def drop_graph(self, graph_name: str):
        """그래프 삭제"""
        self.cursor.execute(f"SELECT drop_graph('{graph_name}', true);")
        print(f"그래프 '{graph_name}' 삭제 완료")

    def cypher(self, graph_name: str, query: str, params: Dict = None) -> List:
        """Cypher 쿼리 실행"""
        sql = f"SELECT * FROM cypher('{graph_name}', $$ {query} $$) AS (result agtype);"
        self.cursor.execute(sql)
        results = self.cursor.fetchall()
        return results

    def add_node(self, graph_name: str, label: str, properties: Dict):
        """노드 추가"""
        props_str = json.dumps(properties, ensure_ascii=False)
        query = f"CREATE (n:{label} {props_str}) RETURN n"
        return self.cypher(graph_name, query)

    def add_edge(self, graph_name: str, from_label: str, from_props: Dict,
                 rel_type: str, to_label: str, to_props: Dict, rel_props: Dict = None):
        """엣지 추가"""
        rel_props_str = json.dumps(rel_props, ensure_ascii=False) if rel_props else ""
        from_match = json.dumps(from_props, ensure_ascii=False)
        to_match = json.dumps(to_props, ensure_ascii=False)
        query = f"""
        MATCH (a:{from_label} {from_match}), (b:{to_label} {to_match})
        CREATE (a)-[r:{rel_type} {rel_props_str}]->(b)
        RETURN r
        """
        return self.cypher(graph_name, query)

    def query_neighbors(self, graph_name: str, node_name: str, max_hops: int = 2) -> List:
        """이웃 노드 검색"""
        query = f"""
        MATCH path = (n {{name: '{node_name}'}})-[*1..{max_hops}]-(m)
        RETURN path
        """
        return self.cypher(graph_name, query)

    def close(self):
        self.cursor.close()
        self.conn.close()


print("ApacheAGE 클래스 정의 완료")
print()
print("사용 예시:")
print('  age = ApacheAGE(host="localhost", dbname="agedb")')
print('  age.create_graph("my_ontology")')
print('  age.add_node("my_ontology", "Person", {"name": "이재용", "role": "회장"})')

In [ ]:
# Cypher 쿼리 기초 - 주요 패턴 정리

cypher_examples = {
    "노드 생성": """
CREATE (n:Person {name: '이재용', role: '회장'})
RETURN n
""",
    "관계 생성": """
MATCH (a:Person {name: '이재용'}), (b:Company {name: '삼성전자'})
CREATE (a)-[:WORKS_AT {since: 2020}]->(b)
RETURN a, b
""",
    "단순 검색": """
MATCH (p:Person)-[:WORKS_AT]->(c:Company)
WHERE c.name = '삼성전자'
RETURN p.name AS employee, c.name AS company
""",
    "멀티홉 검색": """
MATCH (p:Person)-[:WORKS_AT]->(c:Company)-[:PRODUCES]->(prod:Product)
WHERE prod.category = '반도체'
RETURN p.name, c.name, prod.name
""",
    "경로 탐색": """
MATCH path = shortestPath(
  (a:Person {name: '이재용'})-[*]-(b:Product {name: 'HBM'})
)
RETURN path
""",
    "집계 쿼리": """
MATCH (c:Company)-[:PRODUCES]->(p:Product)
RETURN c.name AS company, count(p) AS product_count
ORDER BY product_count DESC
"""
}

print("=== Cypher 쿼리 기초 ===\n")
for title, query in cypher_examples.items():
    print(f"--- {title} ---")
    print(query)

---

## 5️⃣ 온톨로지 RAG 실습

### 전체 파이프라인

```
┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐    ┌──────────────────┐
│  도메인 온톨로지   │ →  │ AGE 그래프 구축   │ →  │ 온톨로지 기반 추론 │ →  │    LLM 연동      │
│  스키마 설계       │    │ 노드/엣지 생성    │    │ 스키마 활용 쿼리   │    │  답변 생성       │
└──────────────────┘    └──────────────────┘    └──────────────────┘    └──────────────────┘
```

In [ ]:
# Step 1: 도메인 온톨로지 설계 (의료 도메인 예시)

medical_ontology = OntologySchema("의료 온톨로지")

# 클래스 정의
medical_ontology.add_class("Disease", properties={"name": "string", "icd_code": "string", "severity": "string"})
medical_ontology.add_class("Symptom", properties={"name": "string", "body_part": "string"})
medical_ontology.add_class("Drug", properties={"name": "string", "type": "string", "dosage": "string"})
medical_ontology.add_class("Treatment", properties={"name": "string", "method": "string"})
medical_ontology.add_class("Patient", properties={"id": "string", "age": "int", "gender": "string"})
medical_ontology.add_class("Doctor", properties={"name": "string", "specialty": "string"})

# 관계 정의
medical_ontology.add_relation("hasSymptom", "Disease", "Symptom", "one-to-many")
medical_ontology.add_relation("treatedBy", "Disease", "Treatment", "many-to-many")
medical_ontology.add_relation("usesDrug", "Treatment", "Drug", "many-to-many")
medical_ontology.add_relation("contraindicated", "Drug", "Drug", "many-to-many")
medical_ontology.add_relation("diagnosedWith", "Patient", "Disease", "many-to-many")
medical_ontology.add_relation("prescribes", "Doctor", "Drug", "many-to-many")
medical_ontology.add_relation("treats", "Doctor", "Patient", "many-to-many")
medical_ontology.add_relation("causesSideEffect", "Drug", "Symptom", "one-to-many")

# 제약조건
medical_ontology.add_constraint("contraindicated 관계에 있는 두 Drug은 동시 처방 불가")
medical_ontology.add_constraint("Patient는 반드시 하나 이상의 Doctor의 treats 관계를 가져야 한다")
medical_ontology.add_constraint("Drug의 causesSideEffect가 Patient의 기존 Symptom과 겹치면 경고")

medical_ontology.display()

In [ ]:
# Step 2: AGE 그래프 구축 (의료 도메인 데이터)

medical_data = {
    "nodes": [
        # 질병
        {"label": "Disease", "props": {"name": "고혈압", "icd_code": "I10", "severity": "만성"}},
        {"label": "Disease", "props": {"name": "당뇨병", "icd_code": "E11", "severity": "만성"}},
        {"label": "Disease", "props": {"name": "폐렴", "icd_code": "J18", "severity": "급성"}},
        # 증상
        {"label": "Symptom", "props": {"name": "두통", "body_part": "머리"}},
        {"label": "Symptom", "props": {"name": "어지러움", "body_part": "전신"}},
        {"label": "Symptom", "props": {"name": "다음다뇨", "body_part": "전신"}},
        {"label": "Symptom", "props": {"name": "기침", "body_part": "호흡기"}},
        {"label": "Symptom", "props": {"name": "발열", "body_part": "전신"}},
        # 약물
        {"label": "Drug", "props": {"name": "암로디핀", "type": "CCB", "dosage": "5mg"}},
        {"label": "Drug", "props": {"name": "메트포르민", "type": "비구아니드", "dosage": "500mg"}},
        {"label": "Drug", "props": {"name": "아목시실린", "type": "항생제", "dosage": "250mg"}},
        # 치료법
        {"label": "Treatment", "props": {"name": "항고혈압치료", "method": "약물치료"}},
        {"label": "Treatment", "props": {"name": "혈당조절치료", "method": "약물+식이"}},
        {"label": "Treatment", "props": {"name": "항생제치료", "method": "약물치료"}},
    ],
    "edges": [
        # 질병-증상
        {"from": "고혈압", "rel": "HAS_SYMPTOM", "to": "두통"},
        {"from": "고혈압", "rel": "HAS_SYMPTOM", "to": "어지러움"},
        {"from": "당뇨병", "rel": "HAS_SYMPTOM", "to": "다음다뇨"},
        {"from": "폐렴", "rel": "HAS_SYMPTOM", "to": "기침"},
        {"from": "폐렴", "rel": "HAS_SYMPTOM", "to": "발열"},
        # 질병-치료
        {"from": "고혈압", "rel": "TREATED_BY", "to": "항고혈압치료"},
        {"from": "당뇨병", "rel": "TREATED_BY", "to": "혈당조절치료"},
        {"from": "폐렴", "rel": "TREATED_BY", "to": "항생제치료"},
        # 치료-약물
        {"from": "항고혈압치료", "rel": "USES_DRUG", "to": "암로디핀"},
        {"from": "혈당조절치료", "rel": "USES_DRUG", "to": "메트포르민"},
        {"from": "항생제치료", "rel": "USES_DRUG", "to": "아목시실린"},
    ]
}


def build_age_graph(age_client, graph_name: str, data: Dict):
    """Apache AGE에 온톨로지 기반 그래프를 구축합니다."""
    age_client.create_graph(graph_name)

    # 노드 생성
    for node in data["nodes"]:
        age_client.add_node(graph_name, node["label"], node["props"])
        print(f"  노드: [{node['label']}] {node['props']['name']}")

    # 엣지 생성
    for edge in data["edges"]:
        print(f"  관계: {edge['from']} --[{edge['rel']}]--> {edge['to']}")

    print(f"\n그래프 '{graph_name}' 구축 완료")
    print(f"  노드: {len(data['nodes'])}개, 엣지: {len(data['edges'])}개")


# AGE 실행 중일 때:
# age = ApacheAGE(host="localhost", dbname="agedb")
# build_age_graph(age, "medical_ontology", medical_data)

print("의료 도메인 그래프 데이터 준비 완료")
print(f"  노드: {len(medical_data['nodes'])}개")
print(f"  엣지: {len(medical_data['edges'])}개")

In [ ]:
# Step 3: 온톨로지 기반 추론 엔진

class OntologyReasoner:
    """온톨로지 스키마를 활용한 추론 엔진"""

    def __init__(self, ontology: OntologySchema, graph_data: Dict):
        self.ontology = ontology
        self.graph = self._build_internal_graph(graph_data)

    def _build_internal_graph(self, data: Dict) -> nx.DiGraph:
        """내부 NetworkX 그래프 구축"""
        G = nx.DiGraph()
        for node in data["nodes"]:
            G.add_node(node["props"]["name"], label=node["label"], **node["props"])
        for edge in data["edges"]:
            G.add_edge(edge["from"], edge["to"], relation=edge["rel"])
        return G

    def find_by_class(self, class_name: str) -> List[str]:
        """특정 클래스의 모든 인스턴스 검색"""
        return [n for n, d in self.graph.nodes(data=True) if d.get("label") == class_name]

    def find_related(self, entity: str, relation: str = None, max_hops: int = 2) -> List[Dict]:
        """엔티티의 관련 노드 검색 (관계 타입 필터링)"""
        results = []
        visited = set()

        def traverse(node, depth):
            if depth > max_hops or node in visited:
                return
            visited.add(node)
            for _, target, data in self.graph.out_edges(node, data=True):
                if relation is None or data["relation"] == relation:
                    results.append({
                        "from": node, "relation": data["relation"],
                        "to": target, "depth": depth
                    })
                traverse(target, depth + 1)

        traverse(entity, 1)
        return results

    def check_constraint(self, entity: str, action: str) -> List[str]:
        """제약조건 검증 (간단한 규칙 기반)"""
        warnings = []
        # 예: 약물 상호작용 검사
        if action == "prescribe":
            current_drugs = [r["to"] for r in self.find_related(entity, "USES_DRUG")]
            for drug in current_drugs:
                contra = self.find_related(drug, "CONTRAINDICATED")
                for c in contra:
                    if c["to"] in current_drugs:
                        warnings.append(f"경고: {drug}와 {c['to']}는 병용 금기입니다!")
        return warnings

    def ontology_guided_query(self, question: str) -> Dict:
        """온톨로지 스키마를 활용한 구조화된 쿼리"""
        # 질문에서 클래스/관계를 매핑
        result = {
            "question": question,
            "identified_classes": [],
            "identified_relations": [],
            "query_path": [],
            "results": []
        }

        # 클래스 매핑 (간단한 키워드 기반)
        class_keywords = {
            "Disease": ["질병", "질환", "병"],
            "Symptom": ["증상", "증세"],
            "Drug": ["약", "약물", "처방"],
            "Treatment": ["치료", "요법"],
        }

        for cls, keywords in class_keywords.items():
            if any(kw in question for kw in keywords):
                result["identified_classes"].append(cls)

        return result


# 추론 엔진 테스트
reasoner = OntologyReasoner(medical_ontology, medical_data)

# 질병별 관련 정보 검색
print("=== 온톨로지 기반 추론 테스트 ===\n")

print("[Disease 클래스 인스턴스]")
diseases = reasoner.find_by_class("Disease")
for d in diseases:
    print(f"  - {d}")

print("\n['고혈압' 관련 정보 (2홉)]")
related = reasoner.find_related("고혈압", max_hops=2)
for r in related:
    indent = "  " * r["depth"]
    print(f"{indent}{r['from']} --[{r['relation']}]--> {r['to']}")

In [ ]:
# Step 4: 온톨로지 RAG - LLM 연동

from openai import OpenAI


class OntologyRAG:
    """온톨로지 기반 RAG 시스템"""

    def __init__(self, ontology: OntologySchema, reasoner: OntologyReasoner):
        self.ontology = ontology
        self.reasoner = reasoner
        self.client = OpenAI()

    def _build_schema_context(self) -> str:
        """온톨로지 스키마를 컨텍스트로 변환"""
        ctx = f"도메인: {self.ontology.name}\n\n"
        ctx += "클래스:\n"
        for cls_name, cls_info in self.ontology.classes.items():
            parent = f" (상위: {cls_info['parent']})" if cls_info['parent'] else ""
            ctx += f"  - {cls_name}{parent}: {cls_info['properties']}\n"
        ctx += "\n관계:\n"
        for rel_name, rel_info in self.ontology.relations.items():
            ctx += f"  - {rel_info['domain']} --[{rel_name}]--> {rel_info['range']}\n"
        ctx += "\n제약조건:\n"
        for c in self.ontology.constraints:
            ctx += f"  - {c}\n"
        return ctx

    def _build_graph_context(self, entities: List[str]) -> str:
        """관련 그래프 데이터를 컨텍스트로 변환"""
        ctx = "관련 지식 그래프 데이터:\n"
        for entity in entities:
            related = self.reasoner.find_related(entity, max_hops=2)
            for r in related:
                ctx += f"  ({r['from']}) --[{r['relation']}]--> ({r['to']})\n"
        return ctx

    def query(self, question: str, entities: List[str]) -> str:
        """온톨로지 RAG 질의"""
        schema_ctx = self._build_schema_context()
        graph_ctx = self._build_graph_context(entities)

        prompt = f"""당신은 의료 도메인 전문가입니다.
다음 온톨로지 스키마와 지식 그래프 데이터를 참고하여 질문에 정확하게 답변하세요.

=== 온톨로지 스키마 ===
{schema_ctx}

=== 지식 그래프 데이터 ===
{graph_ctx}

질문: {question}

온톨로지의 클래스와 관계 정의를 활용하여 체계적으로 답변하세요.
제약조건을 반드시 확인하고, 위반 사항이 있으면 경고하세요."""

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.choices[0].message.content


# 사용 예시
# rag = OntologyRAG(medical_ontology, reasoner)
# answer = rag.query("고혈압 환자에게 처방할 수 있는 약물과 주의사항은?", ["고혈압"])
# print(answer)

print("OntologyRAG 클래스 정의 완료")
print()
print("온톨로지 RAG 동작 흐름:")
print("  1. 질문 분석 → 관련 엔티티 추출")
print("  2. 온톨로지 스키마 컨텍스트 생성")
print("  3. 그래프 탐색으로 관련 데이터 수집")
print("  4. 제약조건 검증")
print("  5. LLM에 스키마 + 데이터 전달 → 답변 생성")

In [ ]:
# 온톨로지 기반 추론 시나리오: 약물 상호작용 검사

def check_drug_interaction(reasoner: OntologyReasoner, disease: str, new_drug: str):
    """환자의 질환에 사용 중인 약물과 새 약물 간 상호작용을 검사합니다."""
    print(f"=== 약물 상호작용 검사 ===")
    print(f"  질환: {disease}")
    print(f"  추가 처방 약물: {new_drug}")
    print()

    # 질환 → 치료 → 기존 약물 경로 추출
    treatments = reasoner.find_related(disease, "TREATED_BY", max_hops=1)
    print("  [현재 치료 경로]")
    for t in treatments:
        print(f"    {t['from']} --[{t['relation']}]--> {t['to']}")
        drugs = reasoner.find_related(t["to"], "USES_DRUG", max_hops=1)
        for d in drugs:
            print(f"      {d['from']} --[{d['relation']}]--> {d['to']}")

    # 제약조건 검사
    warnings = reasoner.check_constraint(disease, "prescribe")
    if warnings:
        print("\n  [경고]")
        for w in warnings:
            print(f"    {w}")
    else:
        print("\n  [결과] 약물 상호작용 문제 없음")


# 테스트
check_drug_interaction(reasoner, "고혈압", "메트포르민")

---

## 6️⃣ 실전 응용: 도메인별 온톨로지 RAG 활용 사례

### 의료 도메인

```
활용 사례:
  - 약물 상호작용 검사 (Drug-Drug Interaction)
  - 증상 기반 질병 진단 보조
  - 치료 프로토콜 추천
  - 의료 문서 자동 분류

온톨로지 예: SNOMED CT, ICD-11, MeSH
```

### 법률 도메인

```
활용 사례:
  - 법률 조항 간 관계 분석
  - 판례 기반 추론 (유사 판례 검색)
  - 계약서 자동 검토 (조항 충돌 탐지)
  - 규정 준수 확인

온톨로지 클래스: 법률, 조항, 판례, 법원, 당사자, 판결
핵심 관계: 인용(cites), 개정(amends), 적용(appliesTo)
```

### 금융 도메인

```
활용 사례:
  - 기업 관계 분석 (지분, 계열사)
  - 사기 탐지 (이상 거래 패턴)
  - 리스크 전파 분석
  - 규제 영향 분석

온톨로지 클래스: 기업, 주식, 펀드, 규제, 거래, 계좌
핵심 관계: 소유(owns), 거래(trades), 보증(guarantees)
```

In [ ]:
# 법률 도메인 온톨로지 예시

legal_ontology = OntologySchema("법률 온톨로지")

legal_ontology.add_class("Law", properties={"name": "string", "enacted_date": "date", "status": "string"})
legal_ontology.add_class("Article", properties={"number": "string", "content": "string"})
legal_ontology.add_class("Case", properties={"case_number": "string", "date": "date", "court": "string"})
legal_ontology.add_class("Party", properties={"name": "string", "role": "string"})
legal_ontology.add_class("Verdict", properties={"result": "string", "summary": "string"})

legal_ontology.add_relation("contains", "Law", "Article", "one-to-many")
legal_ontology.add_relation("cites", "Case", "Article", "many-to-many")
legal_ontology.add_relation("involvedIn", "Party", "Case", "many-to-many")
legal_ontology.add_relation("hasVerdict", "Case", "Verdict", "one-to-one")
legal_ontology.add_relation("amends", "Law", "Law", "many-to-many")
legal_ontology.add_relation("precedentOf", "Case", "Case", "many-to-many")

legal_ontology.add_constraint("Case는 반드시 하나의 Verdict를 가져야 한다")
legal_ontology.add_constraint("amends 관계에서 개정법은 반드시 원법보다 나중에 제정되어야 한다")

legal_ontology.display()

In [ ]:
# 금융 도메인 온톨로지 예시

finance_ontology = OntologySchema("금융 온톨로지")

finance_ontology.add_class("Corporation", properties={"name": "string", "ticker": "string", "sector": "string"})
finance_ontology.add_class("Stock", properties={"ticker": "string", "price": "float", "market_cap": "float"})
finance_ontology.add_class("Fund", properties={"name": "string", "type": "string", "aum": "float"})
finance_ontology.add_class("Regulation", properties={"name": "string", "authority": "string"})
finance_ontology.add_class("Transaction", properties={"id": "string", "amount": "float", "date": "date"})

finance_ontology.add_relation("owns", "Corporation", "Corporation", "many-to-many")
finance_ontology.add_relation("issues", "Corporation", "Stock", "one-to-one")
finance_ontology.add_relation("holdsPosition", "Fund", "Stock", "many-to-many")
finance_ontology.add_relation("regulatedBy", "Corporation", "Regulation", "many-to-many")
finance_ontology.add_relation("executes", "Corporation", "Transaction", "one-to-many")
finance_ontology.add_relation("subsidiaryOf", "Corporation", "Corporation", "many-to-one")

finance_ontology.add_constraint("subsidiaryOf 관계에서 순환 참조는 허용되지 않는다")
finance_ontology.add_constraint("owns 비율의 합은 100%를 초과할 수 없다")

finance_ontology.display()

print("\n" + "=" * 60)
print("도메인별 온톨로지 설계 완료")
print("  - 의료: 질병-증상-약물-치료 관계 모델링")
print("  - 법률: 법률-조항-판례-판결 관계 모델링")
print("  - 금융: 기업-주식-펀드-규제 관계 모델링")

---

## 정리

### 핵심 요약

| 항목 | 그래프 RAG | 온톨로지 RAG |
|------|------------|-------------|
| 스키마 | 암묵적 (데이터 주도) | 명시적 (도메인 전문가 설계) |
| 추론 | 경로 탐색 | 논리적 추론 (상속, 대칭, 전이) |
| 검증 | 제한적 | 제약조건 기반 검증 |
| 구축 비용 | 중간 | 높음 (도메인 전문성 필요) |
| 적합 도메인 | 일반 관계 질문 | 전문 도메인 (의료, 법률, 금융) |

### 실무 적용 가이드

1. **단순 QA** → 벡터 RAG로 충분
2. **관계/경로 질문** → 그래프 RAG 추천
3. **도메인 전문 추론** → 온톨로지 RAG 필요
4. **최적 방안** → 하이브리드 (벡터 + 그래프 + 온톨로지)

In [ ]:
print("=" * 60)
print("온톨로지 RAG 실습 완료!")
print("=" * 60)
print()
print("학습 내용 정리:")
print("  1. 온톨로지 개념: 클래스, 프로퍼티, 관계, 제약조건")
print("  2. 벡터 RAG vs 그래프 RAG vs 온톨로지 RAG 비교")
print("  3. Apache AGE: PostgreSQL 기반 그래프 DB")
print("  4. Cypher 쿼리 기초 및 AGE 사용법")
print("  5. 온톨로지 RAG 파이프라인 구현")
print("  6. 도메인별 활용 사례: 의료, 법률, 금융")